#Análisis de Personas Políticamente Expuestas (PEP) - Uruguay

**Autor:** Federico Ramos  
**Fecha:** Diciembre 2025  
**Formación:** Analista de Datos Jr. - Fundación Telefónica/UCU  
**Objetivo:** Análisis exploratorio para prevención de lavado de activos según Ley 19.574



# 📦 PASO 1: SETUP Y CARGA DE DATOS


In [ ]:
# Importar librerías
import pandas as pd
import numpy as np

# Verificar que funcionan
print("✅ Pandas versión:", pd.__version__)
print("✅ Numpy versión:", np.__version__)
print("\n¡Todo listo para empezar!")

In [ ]:
# Configurar Pandas para mostrar números sin notación científica
pd.options.display.float_format = '{:.2f}'.format

print("✅ Pandas configurado para mostrar números normales")

## 📂 1.1 - Carga del Dataset


In [ ]:
# Subir archivo (va a aparecer un botón para seleccionar)
from google.colab import files
uploaded = files.upload()

# Cargar el CSV
df = pd.read_csv('lista-actualizada-de-pep.csv')
print(f"Dataset cargado: {df.shape}")


# 🔍 PASO 2: EXPLORACIÓN INICIAL

## 2.1 - Primeras Filas del Dataset


In [ ]:
# Ver las primeras 5 filas
print("=" * 60)
print("PRIMERAS 5 FILAS DEL DATASET")
print("=" * 60)
df.head()


## 📋 2.2 - Información General


In [ ]:
# Información del dataset
print("\n" + "=" * 60)
print("INFORMACIÓN GENERAL")
print("=" * 60)
df.info()


## 🔢 2.3 - Valores Únicos


In [ ]:
print("=" * 60)
print("VALORES ÚNICOS POR COLUMNA")
print("=" * 60)

print(f"CI:        {df['CI'].nunique()} valores únicos")
print(f"NOMBRE:    {df['NOMBRE'].nunique()} valores únicos")
print(f"CARGO:     {df['CARGO'].nunique()} valores únicos")
print(f"ORGANISMO: {df['ORGANISMO'].nunique()} valores únicos")


## 📊 2.4 - Estadísticas Descriptivas


In [ ]:
# Estadísticas descriptivas
print("\n" + "=" * 60)
print("ESTADÍSTICAS DESCRIPTIVAS")
print("=" * 60)
df.describe()


# 🧹 PASO 3: LIMPIEZA Y VERIFICACIÓN

## 🔍 3.1 - Verificación de Duplicados


In [ ]:
print("=" * 60)
print("VERIFICACIÓN DE DUPLICADOS")
print("=" * 60)

# ¿Hay filas completamente duplicadas?
duplicados_totales = df.duplicated().sum()
print(f"\nFilas duplicadas (completas): {duplicados_totales}")

if duplicados_totales > 0:
    print("⚠️  Hay duplicados que eliminar")
else:
    print("✅ No hay filas duplicadas")


## 🆔 3.2 - Verificación de CIs Duplicadas


In [ ]:
# ¿Hay CIs repetidas? (esto sería raro, cada persona tiene 1 CI)
duplicados_ci = df['CI'].duplicated().sum()
print(f"\nCIs duplicadas: {duplicados_ci}")

if duplicados_ci > 0:
    print("⚠️  Algunas personas aparecen más de una vez")
    print("\nMostrando personas con CI duplicada:")
    print(df[df['CI'].duplicated(keep=False)].sort_values('CI'))
else:
    print("✅ Cada CI aparece solo una vez")


## 👥 3.3 - Nombres Duplicados (Homónimos)


In [ ]:
# ¿Quiénes tienen el mismo nombre?
duplicados_nombre = df['NOMBRE'].duplicated().sum()
print(f"\nNombres duplicados: {duplicados_nombre}")

if duplicados_nombre > 0:
    print("\nPersonas con el mismo nombre:")
    nombres_repetidos = df[df['NOMBRE'].duplicated(keep=False)].sort_values('NOMBRE')
    print(nombres_repetidos[['CI', 'NOMBRE', 'CARGO']])


## ✅ 3.4 - Resumen de Calidad


In [ ]:
print("\n" + "=" * 60)
print("RESUMEN DE CALIDAD DE DATOS")
print("=" * 60)

print(f"✅ Total de registros: {len(df)}")
print(f"✅ Valores nulos: {df.isnull().sum().sum()}")
print(f"✅ Filas duplicadas: {df.duplicated().sum()}")
print(f"✅ CIs duplicadas: {df['CI'].duplicated().sum()}")
print("\n🎯 Conclusión: El dataset está limpio y listo para analizar")


# 📊 PASO 4: ANÁLISIS EXPLORATORIO

## 🏛️ 4.1 - Top 10 Organismos con mas PePs


In [ ]:
print("=" * 60)
print("TOP 10 ORGANISMOS CON MÁS PEPs")
print("=" * 60)

# Contar cuántas personas hay por organismo
top_organismos = df['ORGANISMO'].value_counts().head(10)

# Mostrar con porcentajes
for i, (organismo, cantidad) in enumerate(top_organismos.items(), 1):
    porcentaje = (cantidad / len(df)) * 100
    print(f"{i:2}. {organismo[:45]:45} → {cantidad:3} ({porcentaje:4.1f}%)")


## 🏛️ 4.2 - Top 10 cargos mas comunes de PePs

In [ ]:
print("\n" + "=" * 60)
print("TOP 10 CARGOS MÁS COMUNES")
print("=" * 60)

top_cargos = df['CARGO'].value_counts().head(10)

for i, (cargo, cantidad) in enumerate(top_cargos.items(), 1):
    porcentaje = (cantidad / len(df)) * 100
    print(f"{i:2}. {cargo:30} → {cantidad:3} ({porcentaje:4.1f}%)")


## 🎯 4.3 - Categorización por Nivel de Riesgo (v1)


In [ ]:
print("=" * 60)
print("CREANDO CATEGORÍA: NIVEL DE RIESGO")
print("=" * 60)

# Función para categorizar cargos
def categorizar_cargo(cargo):
    """
    Esta función lee el nombre del cargo y devuelve el nivel de riesgo
    """
    cargo_mayuscula = cargo.upper()  # Convertir a mayúsculas para buscar

    # Alto Riesgo - Poder Ejecutivo/Legislativo
    if any(palabra in cargo_mayuscula for palabra in
           ['MINISTRO', 'SUBSECRETARIO', 'PRESIDENTE', 'VICEPRESIDENTE',
            'INTENDENTE', 'EMBAJADOR', 'SENADOR', 'REPRESENTANTE NACIONAL']):
        return 'Alto - Ejecutivo/Legislativo'

    # Alto Riesgo - Poder Judicial
    elif any(palabra in cargo_mayuscula for palabra in
             ['JUEZ', 'FISCAL', 'MAGISTRADO']):
        return 'Alto - Poder Judicial'

    # Alto Riesgo - Dirección General
    elif any(palabra in cargo_mayuscula for palabra in
             ['DIRECTOR GENERAL', 'DIRECTOR NACIONAL', 'GERENTE GENERAL']):
        return 'Alto - Dirección General'

    # Medio Riesgo - Dirección/Gerencia
    elif any(palabra in cargo_mayuscula for palabra in
             ['DIRECTOR', 'GERENTE', 'SUBDIRECTOR', 'COORDINADOR']):
        return 'Medio - Dirección/Gerencia'

    # Medio Riesgo - Representación Local
    elif any(palabra in cargo_mayuscula for palabra in
             ['EDIL', 'CONCEJAL', 'ALCALDE']):
        return 'Medio - Representación Local'

    # Otros
    else:
        return 'Otros'

# Aplicar la función a todas las filas
df['NIVEL_RIESGO'] = df['CARGO'].apply(categorizar_cargo)

print("✅ Nueva columna 'NIVEL_RIESGO' creada\n")

# Ver cuántas personas hay en cada nivel
print("Distribución por Nivel de Riesgo:")
print("-" * 60)
niveles = df['NIVEL_RIESGO'].value_counts()

for nivel, cantidad in niveles.items():
    porcentaje = (cantidad / len(df)) * 100
    print(f"{nivel:35} → {cantidad:3} ({porcentaje:5.1f}%)")


## ✔️ 4.4 - Verificación de Categorización


In [ ]:
print("=" * 60)
print("VERIFICACIÓN: ¿La categorización funcionó bien?")
print("=" * 60)

# Ver ejemplos de cada categoría
print("\n1. Ejemplos de ALTO RIESGO - Ejecutivo/Legislativo:")
print(df[df['NIVEL_RIESGO'] == 'Alto - Ejecutivo/Legislativo'][['NOMBRE', 'CARGO']].head(3))

print("\n2. Ejemplos de ALTO RIESGO - Poder Judicial:")
print(df[df['NIVEL_RIESGO'] == 'Alto - Poder Judicial'][['NOMBRE', 'CARGO']].head(3))

print("\n3. Ejemplos de MEDIO RIESGO - Representación Local:")
print(df[df['NIVEL_RIESGO'] == 'Medio - Representación Local'][['NOMBRE', 'CARGO']].head(3))

print("\n4. Ejemplos de OTROS:")
print(df[df['NIVEL_RIESGO'] == 'Otros'][['NOMBRE', 'CARGO']].head(3))


## 🏢 4.5 - Categorización por Tipo de Organismo

In [ ]:
print("=" * 60)
print("CREANDO CATEGORÍA: TIPO DE ORGANISMO")
print("=" * 60)

def categorizar_organismo(organismo):
    """
    Clasifica el organismo según su naturaleza
    """
    org_mayuscula = organismo.upper()

    if 'MINISTERIO' in org_mayuscula:
        return 'Ministerios'

    elif 'PODER JUDICIAL' in org_mayuscula or 'FISCALIA' in org_mayuscula or 'TRIBUNAL' in org_mayuscula:
        return 'Poder Judicial'

    elif 'PODER LEGISLATIVO' in org_mayuscula or 'JUNTA DEPARTAMENTAL' in org_mayuscula:
        return 'Poder Legislativo'

    elif 'INTENDENCIA' in org_mayuscula:
        return 'Gobiernos Departamentales'

    elif 'BANCO' in org_mayuscula or 'BPS' in org_mayuscula or 'BSE' in org_mayuscula:
        return 'Instituciones Financieras'

    elif any(empresa in org_mayuscula for empresa in ['ANTEL', 'ANCAP', 'OSE', 'UTE', 'ANP']):
        return 'Empresas Públicas'

    elif 'UNIVERSIDAD' in org_mayuscula or 'ANEP' in org_mayuscula:
        return 'Educación'

    else:
        return 'Otros Organismos'

# Aplicar la función
df['TIPO_ORGANISMO'] = df['ORGANISMO'].apply(categorizar_organismo)

print("✅ Nueva columna 'TIPO_ORGANISMO' creada\n")

# Ver distribución
print("Distribución por Tipo de Organismo:")
print("-" * 60)
tipos = df['TIPO_ORGANISMO'].value_counts()

for tipo, cantidad in tipos.items():
    porcentaje = (cantidad / len(df)) * 100
    print(f"{tipo:30} → {cantidad:3} ({porcentaje:5.1f}%)")


## 🔍 4.6 - Investigación: Errores Detectados


In [ ]:
print("=" * 60)
print("INVESTIGANDO: ¿Qué pasó con Carolina Ache?")
print("=" * 60)

# Buscar por nombre
carolina = df[df['NOMBRE'].str.contains('ACHE', case=False)]
print(carolina[['NOMBRE', 'CARGO', 'NIVEL_RIESGO']])


## 📋 4.7 - Revisión de Casos "Otros"


In [ ]:
print("\n" + "=" * 60)
print("TODOS LOS CARGOS CLASIFICADOS COMO 'OTROS'")
print("=" * 60)

otros = df[df['NIVEL_RIESGO'] == 'Otros'][['NOMBRE', 'CARGO', 'ORGANISMO']]
print(f"\nTotal de 'Otros': {len(otros)}\n")

# Mostrar los primeros 20 para revisar
print("Primeros 20 casos:")
print(otros.head(20).to_string())


## 🔧 4.8 - Categorización Mejorada (v2)

**Iteración:** crear → verificar → mejorar

In [ ]:
print("=" * 60)
print("MEJORANDO LA CATEGORIZACIÓN")
print("=" * 60)

def categorizar_cargo_v2(cargo):
    """
    Versión mejorada con más casos
    """
    cargo_mayuscula = cargo.upper()

    # Alto Riesgo - Poder Ejecutivo/Legislativo
    # IMPORTANTE: Agregamos versiones femeninas y más cargos
    if any(palabra in cargo_mayuscula for palabra in
           ['MINISTRO', 'MINISTRA', 'SUBSECRETARIO', 'SUBSECRETARIA',
            'PRESIDENTE', 'PRESIDENTA', 'VICEPRESIDENTE', 'VICEPRESIDENTA',
            'INTENDENTE', 'INTENDENTA', 'EMBAJADOR', 'EMBAJADORA',
            'SENADOR', 'SENADORA', 'REPRESENTANTE NACIONAL',
            'REPRESENTANTE PERMANENTE', 'GENERAL', 'ALMIRANTE']):
        return 'Alto - Ejecutivo/Legislativo'

    # Alto Riesgo - Poder Judicial
    elif any(palabra in cargo_mayuscula for palabra in
             ['JUEZ', 'JUEZA', 'FISCAL', 'MAGISTRADO', 'MAGISTRADA']):
        return 'Alto - Poder Judicial'

    # Alto Riesgo - Dirección General
    elif any(palabra in cargo_mayuscula for palabra in
             ['DIRECTOR GENERAL', 'DIRECTORA GENERAL',
              'DIRECTOR NACIONAL', 'DIRECTORA NACIONAL',
              'GERENTE GENERAL']):
        return 'Alto - Dirección General'

    # Medio-Alto - Cargos diplomáticos y policiales
    elif any(palabra in cargo_mayuscula for palabra in
             ['CONSUL GENERAL', 'CÓNSUL GENERAL',
              'JEFE DE POLICIA', 'SUBJEFE DE POLICIA',
              'ENCARGADO DE NEGOCIOS', 'COMISARIO GENERAL']):
        return 'Medio-Alto - Diplomático/Policial'

    # Medio Riesgo - Dirección/Gerencia
    elif any(palabra in cargo_mayuscula for palabra in
             ['DIRECTOR', 'DIRECTORA', 'GERENTE',
              'SUBDIRECTOR', 'SUBDIRECTORA', 'COORDINADOR', 'COORDINADORA']):
        return 'Medio - Dirección/Gerencia'

    # Medio Riesgo - Representación Local
    elif any(palabra in cargo_mayuscula for palabra in
             ['EDIL', 'CONCEJAL', 'ALCALDE', 'ALCALDESA']):
        return 'Medio - Representación Local'

    # Otros
    else:
        return 'Otros'

# Aplicar la nueva función
df['NIVEL_RIESGO'] = df['CARGO'].apply(categorizar_cargo_v2)

print("✅ Categorización actualizada\n")

# Ver cómo cambió
print("Nueva distribución:")
print("-" * 60)
niveles = df['NIVEL_RIESGO'].value_counts()

for nivel, cantidad in niveles.items():
    porcentaje = (cantidad / len(df)) * 100
    print(f"{nivel:35} → {cantidad:3} ({porcentaje:5.1f}%)")

print("\n¿Cuántos 'Otros' quedan ahora?")
print(f"→ {len(df[df['NIVEL_RIESGO'] == 'Otros'])} personas")


## ✅ 4.9 - Verificación de Correcciones


In [ ]:
# Verificar que Carolina Ache cambió
print("\n" + "=" * 60)
print("VERIFICACIÓN: ¿Se corrigió Carolina Ache?")
print("=" * 60)

carolina = df[df['NOMBRE'].str.contains('ACHE', case=False)]
print(carolina[['NOMBRE', 'CARGO', 'NIVEL_RIESGO']])


## 📅 4.10 - Estimación de Rango Etario


In [ ]:
print("=" * 60)
print("CREANDO CATEGORÍA: RANGO DE EDAD (ESTIMADO)")
print("=" * 60)

def estimar_edad(ci):
    """
    Estima rango de edad según la CI uruguaya
    CIs más bajas = personas mayores
    """
    if ci < 20000000:
        return 'Mayor 50 años'
    elif ci < 40000000:
        return '35-50 años'
    else:
        return 'Menor 35 años'

# Aplicar la función
df['RANGO_EDAD'] = df['CI'].apply(estimar_edad)

print("✅ Nueva columna 'RANGO_EDAD' creada\n")

# Ver distribución
print("Distribución por Rango de Edad:")
print("-" * 60)
edades = df['RANGO_EDAD'].value_counts().sort_index()

for edad, cantidad in edades.items():
    porcentaje = (cantidad / len(df)) * 100
    print(f"{edad:15} → {cantidad:3} ({porcentaje:5.1f}%)")


In [ ]:
print("\n" + "=" * 60)
print("EJEMPLOS: CI vs EDAD ESTIMADA")
print("=" * 60)

# Mostrar algunos casos para ver si tiene sentido
ejemplos = df[['CI', 'NOMBRE', 'RANGO_EDAD']].sample(10).sort_values('CI')
print(ejemplos.to_string(index=False))


---
# 🔥 PASO 5: ANÁLISIS CRUZADO

## 🚨 5.1 - Alto Riesgo por Organismo


In [ ]:
print("=" * 60)
print("ANÁLISIS CRUZADO 1: Alto Riesgo por Organismo")
print("=" * 60)

# Filtrar solo Alto Riesgo
alto_riesgo = df[df['NIVEL_RIESGO'].str.contains('Alto')]

# Top 10 organismos con más PEPs de Alto Riesgo
top_alto = alto_riesgo['ORGANISMO'].value_counts().head(10)

print(f"\nTotal de PEPs de Alto Riesgo: {len(alto_riesgo)} ({len(alto_riesgo)/len(df)*100:.1f}%)\n")
print("Top 10 organismos con PEPs de Alto Riesgo:")
print("-" * 60)

for i, (org, cant) in enumerate(top_alto.items(), 1):
    porcentaje = (cant / len(alto_riesgo)) * 100
    print(f"{i:2}. {org[:50]:50} {cant:3} ({porcentaje:4.1f}%)")


## 🏦 5.2 - PEPs en Instituciones Financieras


In [ ]:
print("\n" + "=" * 60)
print("ANÁLISIS CRUZADO 2: PEPs en Instituciones Financieras")
print("=" * 60)

# Filtrar instituciones financieras
fin = df[df['TIPO_ORGANISMO'] == 'Instituciones Financieras']

print(f"\nTotal de PEPs en instituciones financieras: {len(fin)}\n")

# ¿Qué nivel de riesgo tienen?
print("Distribución por Nivel de Riesgo:")
print("-" * 60)
for nivel, cant in fin['NIVEL_RIESGO'].value_counts().items():
    porcentaje = (cant / len(fin)) * 100
    print(f"{nivel:35} → {cant:2} ({porcentaje:5.1f}%)")

print("\n¿En qué instituciones están?")
print("-" * 60)
for inst, cant in fin['ORGANISMO'].value_counts().items():
    print(f"  {inst}: {cant}")


In [ ]:
print("\n" + "=" * 60)
print("ANÁLISIS CRUZADO 3: Edad de PEPs de Alto Riesgo")
print("=" * 60)

# Filtrar Alto Riesgo
alto_riesgo = df[df['NIVEL_RIESGO'].str.contains('Alto')]

print("Distribución etaria de PEPs de Alto Riesgo:")
print("-" * 60)

for edad, cant in alto_riesgo['RANGO_EDAD'].value_counts().sort_index().items():
    porcentaje = (cant / len(alto_riesgo)) * 100
    print(f"{edad:15} → {cant:3} ({porcentaje:5.1f}%)")


In [ ]:
print("\n" + "=" * 60)
print("TABLA RESUMEN EJECUTIVO")
print("=" * 60)

# Crear tabla cruzada: Tipo de Organismo vs Nivel de Riesgo
tabla_resumen = pd.crosstab(
    df['TIPO_ORGANISMO'],
    df['NIVEL_RIESGO'],
    margins=True,  # Agregar totales
    margins_name='TOTAL'
)

print("\nCantidad de PEPs por Tipo de Organismo y Nivel de Riesgo:")
print(tabla_resumen)

# Guardarlo en CSV para usar después
tabla_resumen.to_csv('tabla_cruzada_pep.csv')
print("\n✅ Tabla guardada en: tabla_cruzada_pep.csv")


## 🚨 5.3 - CASOS CRÍTICOS DE: Alto Riesgo en Finanzas

**Máxima prioridad para Compliance**


In [ ]:
print("=" * 60)
print("🚨 ANÁLISIS CRÍTICO: Alto Riesgo en Finanzas")
print("=" * 60)

# Filtrar: Alto Riesgo + Instituciones Financieras
criticos = df[
    (df['NIVEL_RIESGO'].str.contains('Alto')) &
    (df['TIPO_ORGANISMO'] == 'Instituciones Financieras')
]

print(f"\n⚠️  TOTAL DE CASOS CRÍTICOS: {len(criticos)}\n")
print("Estos son PEPs de Alto Riesgo que trabajan EN instituciones financieras.")
print("Requieren MÁXIMA diligencia según Ley 19.574\n")
print("=" * 60)

if len(criticos) > 0:
    # Mostrar todos los casos
    print("\nLISTA COMPLETA DE CASOS CRÍTICOS:")
    print("-" * 60)

    for idx, row in criticos.iterrows():
        print(f"\n👤 {row['NOMBRE']}")
        print(f"   📋 Cargo: {row['CARGO']}")
        print(f"   🏛️  Organismo: {row['ORGANISMO']}")
        print(f"   ⚠️  Nivel de Riesgo: {row['NIVEL_RIESGO']}")
        print(f"   📅 Rango Edad: {row['RANGO_EDAD']}")
        print(f"   🆔 CI: {row['CI']}")

    print("\n" + "=" * 60)
    print("💡 RECOMENDACIÓN:")
    print("   • Monitoreo reforzado de transacciones")
    print("   • Actualización trimestral de perfil")
    print("   • Verificación de conflictos de interés")
    print("   • Seguimiento de patrimonio declarado")

    # Guardar lista en CSV
    criticos[['CI', 'NOMBRE', 'CARGO', 'ORGANISMO', 'NIVEL_RIESGO']].to_csv(
        'pep_alto_riesgo_finanzas.csv',
        index=False
    )
    print("\n✅ Lista exportada: pep_alto_riesgo_finanzas.csv")

else:
    print("No se encontraron PEPs de Alto Riesgo en instituciones financieras.")


## 📈 5.4 - Análisis Detallado: Sector Financiero
## 📋 5.5 - Registro: Medio Riesgo en Finanzas



In [ ]:
print("=" * 80)
print("📊 REGISTRO: OTROS PEPs EN INSTITUCIONES FINANCIERAS")
print("=" * 80)

fin_todas = df[df['TIPO_ORGANISMO'] == 'Instituciones Financieras']
# Define fin_resto: personas en instituciones financieras que NO son de 'Alto' riesgo
fin_resto = fin_todas[~fin_todas['NIVEL_RIESGO'].str.contains('Alto')]

# Niveles que quedan (sin los "Alto")
niveles_orden = [
    'Medio-Alto - Diplomático/Policial',
    'Medio - Dirección/Gerencia',
    'Medio - Representación Local',
    'Otros'
]

contador_general = 1

for nivel in niveles_orden:
    personas_nivel = fin_resto[fin_resto['NIVEL_RIESGO'] == nivel]

    if len(personas_nivel) == 0:
        continue

    print("\n" + "█" * 80)
    print(f"║ NIVEL DE RIESGO: {nivel}")
    print(f"║ Total: {len(personas_nivel)} persona(s)")
    print("█" * 80)

    # Ordenar por organismo y nombre
    personas_ordenado = personas_nivel.sort_values(['ORGANISMO', 'NOMBRE'])

    organismo_actual = None

    for idx, row in personas_ordenado.iterrows():
        # Mostrar organismo cuando cambia
        if row['ORGANISMO'] != organismo_actual:
            organismo_actual = row['ORGANISMO']
            print(f"\n┌─ {organismo_actual}")
            print("│")

        # Datos de la persona
        print(f"│  {contador_general:2}. 👤 {row['NOMBRE']}")
        print(f"│      📋 Cargo: {row['CARGO']}")
        print(f"│      🆔 CI: {row['CI']}")
        print(f"│      📅 Edad: {row['RANGO_EDAD']}")
        print("│")

        contador_general += 1

print("\n" + "=" * 80)
print(f"✅ TOTAL (excluyendo Alto Riesgo): {len(fin_resto)} PEPs")
print("=" * 80)

# Exportar solo los 48
columnas = ['CI', 'NOMBRE', 'CARGO', 'ORGANISMO', 'NIVEL_RIESGO', 'RANGO_EDAD']
export = fin_resto[columnas].copy()
export = export.sort_values(['NIVEL_RIESGO', 'ORGANISMO', 'NOMBRE'])
export.to_csv('REGISTRO_48_PEPs_MEDIO_RIESGO_FINANZAS.csv', index=False)

print("\n✅ Archivo: REGISTRO_48_PEPs_MEDIO_RIESGO_FINANZAS.csv")
print("   (Los 48 restantes, sin los 4 críticos)")

---
# 📄 PASO 6: COMUNICACIÓN Y EXPORTACIÓN

## 📝 6.1 - Reporte Ejecutivo Final


In [ ]:
print("=" * 70)
print(" " * 15 + "REPORTE EJECUTIVO FINAL")
print("=" * 70)

# Calcular estadísticas clave
alto_total = len(df[df['NIVEL_RIESGO'].str.contains('Alto')])
medio_total = len(df[df['NIVEL_RIESGO'].str.contains('Medio')])
fin_todas = df[df['TIPO_ORGANISMO'] == 'Instituciones Financieras']
criticos = df[(df['NIVEL_RIESGO'].str.contains('Alto')) &
              (df['TIPO_ORGANISMO'] == 'Instituciones Financieras')]
fin_resto = fin_todas[~fin_todas['NIVEL_RIESGO'].str.contains('Alto')]

resumen = f"""
╔══════════════════════════════════════════════════════════════════════╗
║        ANÁLISIS DE PERSONAS POLÍTICAMENTE EXPUESTAS (PEP)           ║
║                    URUGUAY - DICIEMBRE 2025                          ║
║                 Prevención de Lavado de Activos                      ║
╚══════════════════════════════════════════════════════════════════════╝

📊 DATOS GENERALES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Total de PEPs registrados: {len(df):,}
• Organismos distintos: {df['ORGANISMO'].nunique()}
• Cargos distintos: {df['CARGO'].nunique()}
• Fuente: SENACLAFT - Catálogo de Datos Abiertos
• Marco legal: Ley N° 19.574 (Prevención de Lavado de Activos)

🎯 DISTRIBUCIÓN POR NIVEL DE RIESGO
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""

# Mostrar niveles ordenados
niveles_ordenados = [
    'Alto - Ejecutivo/Legislativo',
    'Alto - Poder Judicial',
    'Alto - Dirección General',
    'Medio-Alto - Diplomático/Policial',
    'Medio - Dirección/Gerencia',
    'Medio - Representación Local',
    'Otros'
]

for nivel in niveles_ordenados:
    cant = len(df[df['NIVEL_RIESGO'] == nivel])
    if cant > 0:
        pct = (cant/len(df)*100)
        resumen += f"• {nivel:40} {cant:3} ({pct:5.1f}%)\n"

resumen += f"\n  → TOTAL ALTO RIESGO: {alto_total} ({alto_total/len(df)*100:.1f}%)\n"
resumen += f"  → TOTAL MEDIO RIESGO: {medio_total} ({medio_total/len(df)*100:.1f}%)\n"

resumen += f"""
🏛️  DISTRIBUCIÓN POR TIPO DE ORGANISMO
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""

for tipo, cant in df['TIPO_ORGANISMO'].value_counts().items():
    pct = (cant/len(df)*100)
    resumen += f"• {tipo:30} {cant:3} ({pct:5.1f}%)\n"

resumen += f"""
👥 PERFIL ETARIO (Estimado por CI)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""

for edad in ['Menor 35 años', '35-50 años', 'Mayor 50 años']:
    cant = len(df[df['RANGO_EDAD'] == edad])
    pct = (cant/len(df)*100)
    resumen += f"• {edad:20} {cant:3} ({pct:5.1f}%)\n"

resumen += f"""
🚨 CASOS CRÍTICOS: Alto Riesgo en Instituciones Financieras
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Total identificados: {len(criticos)}
• Requieren debida diligencia REFORZADA según Art. 8 Ley 19.574

Distribución por institución:
"""

for org in sorted(criticos['ORGANISMO'].unique()):
    cant = len(criticos[criticos['ORGANISMO'] == org])
    # Mostrar cargos específicos
    cargos = criticos[criticos['ORGANISMO'] == org]['CARGO'].tolist()
    org_corto = org.replace('BANCO DE LA REPUBLICA ORIENTAL DEL URUGUAY', 'BROU').replace('BANCO DE SEGUROS DEL ESTADO', 'BSE').replace('BANCO HIPOTECARIO DEL URUGUAY', 'BHU')
    resumen += f"  • {org_corto}: {cant} PEP(s)\n"
    for cargo in cargos:
        resumen += f"      - {cargo}\n"

resumen += f"""
📊 CONTEXTO: Instituciones Financieras (Todos los niveles)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Total de PEPs en sector financiero: {len(fin_todas)} ({len(fin_todas)/len(df)*100:.1f}% del total)
• Alto Riesgo: {len(criticos)} ({len(criticos)/len(fin_todas)*100:.1f}% del sector)
• Medio Riesgo: {len(fin_resto)} ({len(fin_resto)/len(fin_todas)*100:.1f}% del sector)

Distribución por institución:
  • BPS (Previsión Social): {len(fin_todas[fin_todas['ORGANISMO'].str.contains('BPS')])} PEPs
  • BSE (Seguros): {len(fin_todas[fin_todas['ORGANISMO'].str.contains('BSE')])} PEPs
  • BROU (Banca Comercial): {len(fin_todas[fin_todas['ORGANISMO'].str.contains('BROU')])} PEPs
  • BCU (Banca Central): {len(fin_todas[fin_todas['ORGANISMO'].str.contains('BCU')])} PEPs
  • BHU (Hipotecario): {len(fin_todas[fin_todas['ORGANISMO'].str.contains('BHU')])} PEPs

📋 RECOMENDACIONES PARA SUJETOS OBLIGADOS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. PRIORIZACIÓN DE RECURSOS
   • Foco inmediato: {len(criticos)} casos críticos (Alto Riesgo en finanzas)
   • Monitoreo reforzado: {alto_total} PEPs de Alto Riesgo total
   • Seguimiento estándar: {medio_total} PEPs de Medio Riesgo

2. MONITOREO CONTINUO
   • Actualización trimestral desde fuente oficial (catalogodatos.gub.uy)
   • Verificación automática por CI en proceso de onboarding
   • Cross-check con familiares directos (cónyuge, hijos, padres)
   • Alertas por cambios de cargo o institución

3. DEBIDA DILIGENCIA REFORZADA (Art. 8 Ley 19.574)
   Para PEPs de Alto Riesgo:
   • Aprobación de alta dirección para iniciar relación comercial
   • Identificación del origen de fondos y patrimonio
   • Monitoreo continuo de transacciones
   • Revisión anual del perfil de riesgo

   Especial atención a:
   • {len(df[df['NIVEL_RIESGO'] == 'Alto - Poder Judicial'])} PEPs en Poder Judicial
   • {len(df[df['NIVEL_RIESGO'] == 'Alto - Ejecutivo/Legislativo'])} PEPs Ejecutivo/Legislativo
   • {len(criticos)} PEPs de Alto Riesgo DENTRO de instituciones financieras

4. GESTIÓN DOCUMENTAL
   • Mantener expediente actualizado de cada PEP
   • Justificar por escrito decisiones sobre relaciones de alto riesgo
   • Registrar transacciones inusuales en sistema de alertas
   • Reporte mensual a Oficial de Cumplimiento
   • Archivo de evidencia para inspecciones SENACLAFT

5. CAPACITACIÓN DEL PERSONAL
   • Entrenamiento anual en identificación de PEPs
   • Actualización sobre cambios en lista oficial
   • Casos prácticos de señales de alerta
   • Procedimientos de escalamiento

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📁 ARCHIVOS GENERADOS:
   • Dataset completo enriquecido con categorías de análisis
   • Lista de {len(criticos)} casos críticos para seguimiento inmediato
   • Registro de {len(fin_resto)} PEPs de medio riesgo en finanzas
   • Tablas resumen para integración con sistemas KYC/AML

💡 VALOR DEL ANÁLISIS:
   Este estudio permite a las instituciones financieras cumplir eficientemente
   con la normativa de prevención de lavado de activos, priorizando recursos
   en los casos de mayor exposición y automatizando la verificación de PEPs.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Analista: Federico Ramos
Formación: Analista de Datos Jr. - Fundación Telefónica/UCU (300hs)
Fecha: {pd.Timestamp.now().strftime('%d de %B de %Y')}
Herramientas: Python 3.x, Pandas, Google Colab
Fuente datos: SENACLAFT - https://catalogodatos.gub.uy/
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""

print(resumen)

# Guardar en archivo de texto
with open('REPORTE_EJECUTIVO_PEP.txt', 'w', encoding='utf-8')  as f:
    f.write(resumen)

print("\n✅ Reporte guardado: REPORTE_EJECUTIVO_PEP.txt")
print("   Este archivo puede imprimirse o enviarse por email a stakeholders")


---
# ✅ PROYECTO COMPLETADO

📊 **857 PEPs analizados y categorizados**  
🚨 **4 casos críticos identificados**  
📦 **Archivos exportados para uso operativo**

*Federico Ramos - Diciembre 2025*

In [ ]:
import os

print("=" * 60)
print("📦 ARCHIVOS GENERADOS EN ESTE PROYECTO")
print("=" * 60)

archivos_csv = [f for f in os.listdir() if f.endswith('.csv')]
archivos_txt = [f for f in os.listdir() if f.endswith('.txt')]

print("\n📄 Archivos CSV:")
for archivo in sorted(archivos_csv):
    size = os.path.getsize(archivo) / 1024
    print(f"  • {archivo} ({size:.1f} KB)")

print("\n📝 Archivos TXT:")
for archivo in sorted(archivos_txt):
    size = os.path.getsize(archivo) / 1024
    print(f"  • {archivo} ({size:.1f} KB)")

print("\n✅ Proyecto completado exitosamente")
